# Tabular Cleaning — Fixed-Rule Preprocessing Only

In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, r"C:\FYP\src")
from utils.config import TABULAR_RAW_PATH, TABULAR_CLEAN_PATH

pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


## Section 1 — Load Raw Data

In [2]:
df = pd.read_csv(TABULAR_RAW_PATH)

print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Raw shape: (590, 14)
Columns: ['sample_id', 'patient_cohort', 'sample_origin', 'age', 'sex', 'diagnosis', 'stage', 'benign_sample_diagnosis', 'plasma_CA19_9', 'creatinine', 'LYVE1', 'REG1B', 'TFF1', 'REG1A']


,sample_id,patient_cohort,sample_origin,age,sex,diagnosis,stage,benign_sample_diagnosis,plasma_CA19_9,creatinine,LYVE1,REG1B,TFF1,REG1A
0,S1,Cohort1,BPTB,33,F,1,NaN,NaN,11.7,1.83222,0.893219,52.94884,654.282174,1262.000
1,S10,Cohort1,BPTB,81,F,1,NaN,NaN,NaN,0.97266,2.037585,94.46703,209.488250,228.407
2,S100,Cohort2,BPTB,51,M,1,NaN,NaN,7.0,0.78039,0.145589,102.36600,461.141000,NaN
3,S101,Cohort2,BPTB,61,M,1,NaN,NaN,8.0,0.70122,0.002805,60.57900,142.950000,NaN
4,S102,Cohort2,BPTB,62,M,1,NaN,NaN,9.0,0.21489,0.000860,65.54000,41.088000,NaN


## Section 2 — Derive Target Labels

In [3]:
DX_MAP = {1: "Control", 2: "Benign", 3: "PDAC"}
df["dx"] = df["diagnosis"].map(DX_MAP)
df["target_binary"] = (df["diagnosis"] == 3).astype(int)

assert df["dx"].isna().sum() == 0, "unmapped diagnosis code found"

print(df["dx"].value_counts())
print()
print(df["target_binary"].value_counts().rename({0: "not PDAC", 1: "PDAC"}))

dx
Benign     208
PDAC       199
Control    183
Name: count, dtype: int64

target_binary
not PDAC    391
PDAC        199
Name: count, dtype: int64


## Section 3 — Sentinel-Fill `stage` and `benign_sample_diagnosis`

In [4]:
metadata_sentinel = pd.DataFrame(index=df.index)
metadata_sentinel["stage"] = df["stage"].fillna("No Cancer")
metadata_sentinel["benign_sample_diagnosis"] = df["benign_sample_diagnosis"].fillna("Control/PDAC")

check_stage = (metadata_sentinel["stage"] != "No Cancer") == (df["diagnosis"] == 3)
check_benign = (metadata_sentinel["benign_sample_diagnosis"] != "Control/PDAC") == (df["diagnosis"] == 2)

assert check_stage.all(), "stage sentinel-fill does not exactly match diagnosis == 3"
assert check_benign.all(), "benign_sample_diagnosis sentinel-fill does not exactly match diagnosis == 2"

print("Sentinel-fill verification (all 590 rows):")
print(f"  (stage != 'No Cancer') == (diagnosis == 3)                      -> {check_stage.all()}")
print(f"  (benign_sample_diagnosis != 'Control/PDAC') == (diagnosis == 2) -> {check_benign.all()}")

Sentinel-fill verification (all 590 rows):
  (stage != 'No Cancer') == (diagnosis == 3)                      -> True
  (benign_sample_diagnosis != 'Control/PDAC') == (diagnosis == 2) -> True


## Section 4 — Drop `REG1A`

In [5]:
df = df.drop(columns=["REG1A"])
print(f"Dropped REG1A. Shape now: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dropped REG1A. Shape now: (590, 15)
Columns: ['sample_id', 'patient_cohort', 'sample_origin', 'age', 'sex', 'diagnosis', 'stage', 'benign_sample_diagnosis', 'plasma_CA19_9', 'creatinine', 'LYVE1', 'REG1B', 'TFF1', 'dx', 'target_binary']


## Section 5 — Build `FEATURES`

In [6]:
SEX_MAP = {"F": 0, "M": 1}
df["sex_encoded"] = df["sex"].map(SEX_MAP)
assert df["sex_encoded"].isna().sum() == 0, "unmapped sex value found"

FEATURES = ["creatinine", "LYVE1", "REG1B", "TFF1", "plasma_CA19_9", "age", "sex"]

feature_matrix = df[["creatinine", "LYVE1", "REG1B", "TFF1", "plasma_CA19_9", "age", "sex_encoded"]].copy()
feature_matrix.columns = FEATURES

print(f"feature_matrix shape: {feature_matrix.shape}")
print(feature_matrix.dtypes)
print()
print("Missing values per feature (plasma_CA19_9 NaNs are expected and intentional):")
print(feature_matrix.isna().sum())

feature_matrix shape: (590, 7)
creatinine       float64
LYVE1            float64
REG1B            float64
TFF1             float64
plasma_CA19_9    float64
age                int64
sex                int64
dtype: object

Missing values per feature (plasma_CA19_9 NaNs are expected and intentional):
creatinine         0
LYVE1              0
REG1B              0
TFF1               0
plasma_CA19_9    240
age                0
sex                0
dtype: int64


## Section 6 — Build `METADATA`

In [7]:
METADATA = pd.DataFrame({
    "sample_id": df["sample_id"],
    "patient_cohort": df["patient_cohort"],
    "sample_origin": df["sample_origin"],
    "stage": metadata_sentinel["stage"],
    "diagnosis": df["diagnosis"],
})

TARGETS = pd.DataFrame({
    "dx": df["dx"],
    "target_binary": df["target_binary"],
})

assert feature_matrix.index.equals(METADATA.index)
assert feature_matrix.index.equals(TARGETS.index)

print(f"feature_matrix: {feature_matrix.shape}   METADATA: {METADATA.shape}   TARGETS: {TARGETS.shape}")
print("All three frames share the same index -- aligned, never merged.")
METADATA.head()

feature_matrix: (590, 7)   METADATA: (590, 5)   TARGETS: (590, 2)
All three frames share the same index -- aligned, never merged.


,sample_id,patient_cohort,sample_origin,stage,diagnosis
0,S1,Cohort1,BPTB,No Cancer,1
1,S10,Cohort1,BPTB,No Cancer,1
2,S100,Cohort2,BPTB,No Cancer,1
3,S101,Cohort2,BPTB,No Cancer,1
4,S102,Cohort2,BPTB,No Cancer,1


## Write `tabular_clean.csv`

In [8]:
tabular_clean_df = pd.concat([METADATA, TARGETS, feature_matrix], axis=1)

assert len(tabular_clean_df) == 590
assert tabular_clean_df.columns.duplicated().sum() == 0, "unexpected column-name collision"

tabular_clean_df.to_csv(TABULAR_CLEAN_PATH, index=False)
print(f"Wrote {TABULAR_CLEAN_PATH}")
print(f"Shape: {tabular_clean_df.shape}")
print(f"Columns: {list(tabular_clean_df.columns)}")
print(f"plasma_CA19_9 missing (expected, left raw): {tabular_clean_df['plasma_CA19_9'].isna().sum()}")

Wrote C:\FYP\data\processed\tabular_clean.csv
Shape: (590, 14)
Columns: ['sample_id', 'patient_cohort', 'sample_origin', 'stage', 'diagnosis', 'dx', 'target_binary', 'creatinine', 'LYVE1', 'REG1B', 'TFF1', 'plasma_CA19_9', 'age', 'sex']
plasma_CA19_9 missing (expected, left raw): 240
